# PubLayNet Document Layout Detection Benchmark Analysis

## Executive Summary

**Dataset:** PubLayNet - Document Layout Detection  
**Total Samples:** 500 images per phase  
**Task:** Bounding box extraction for document elements (text, title, list, table, figure)  
**Evaluation Metrics:** IoU (Intersection over Union), Precision, Recall, F1-Score, Category Accuracy

## Benchmark Structure

### Phase P-A: OCR Baseline (Pure OCR Models)
- **Models:** Azure Document Intelligence, Mistral Document AI
- **Approach:** Direct OCR with layout detection
- **Purpose:** Establish baseline OCR performance for layout detection

### Phase P-B: VLM Baseline (Generic Prompting)
- **Models:** GPT-5-mini, GPT-5-nano, Claude Sonnet  
- **Prompt:** Generic layout extraction (no domain context)
- **Purpose:** Evaluate general-purpose VLM capabilities for layout detection

### Phase P-C: VLM with Task-Aware Prompting
- **Models:** GPT-5-mini, GPT-5-nano, Claude Sonnet
- **Prompt:** Task-specific instructions for document layout analysis
- **Purpose:** Evaluate impact of task-aware prompting on bounding box extraction

## Key Metrics

- **Precision:** Proportion of predicted boxes that match ground truth (TP / (TP + FP))
- **Recall:** Proportion of ground truth boxes that were detected (TP / (TP + FN))
- **F1-Score:** Harmonic mean of precision and recall
- **Average IoU:** Mean Intersection over Union for matched boxes
- **Category Accuracy:** Proportion of matched boxes with correct category label

## Data Quality

⚠️ **Important:** This notebook identifies and filters out rows with:
- Empty predictions (blank or NaN values)
- Error values in error columns

These rows are excluded from all metric calculations to ensure accurate evaluation.

## Analysis Focus Areas

1. **OCR vs VLM:** Do vision language models outperform specialized OCR for layout detection?
2. **Prompting Impact:** How much does task-aware prompting improve VLM performance?
3. **Category Performance:** Which document elements (text, title, table, figure) are easiest/hardest?
4. **Model Comparison:** Trade-offs between speed and accuracy across models
5. **Error Patterns:** Over-detection vs under-detection analysis

## To Run This Analysis

1. Ensure consolidated data exists in `2_clean/publaynet_full/`
2. This notebook will load P-A, P-B, P-C results and generate:
   - Bounding box evaluation metrics (IoU, Precision, Recall, F1)
   - Category-level performance analysis
   - Model comparison visualizations
   - Sample-level analysis (easiest/hardest samples)
   - Detailed error analysis

## 1. Import Required Libraries

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sys
from pathlib import Path
from typing import List, Dict, Optional, Tuple
import warnings
warnings.filterwarnings('ignore')

# Add parent directory to path for imports
sys.path.insert(0, str(Path.cwd().parent))

# Import evaluation metrics (for any auxiliary metrics needed)
from ocr_vs_vlm.metrics.evaluation_metrics import aggregate_metrics

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("✓ Libraries loaded successfully!")

✓ Libraries loaded successfully!


## 2. Load Dataset and Results

Load the consolidated results from the 2_clean folder.

In [8]:
# Define paths
RESULTS_DIR = Path("../../../2_clean/publaynet_full")

# Check available files
available_files = list(RESULTS_DIR.glob("*.csv"))
print("Available files:")
for f in sorted(available_files):
    print(f"  - {f.name}")

Available files:


In [9]:
# Load results for each phase
phase_dfs = {}

for phase in ['P-A', 'P-B', 'P-C']:
    file_path = RESULTS_DIR / f"{phase}.csv"
    if file_path.exists():
        phase_dfs[phase] = pd.read_csv(file_path)
        print(f"{phase}: {phase_dfs[phase].shape[0]} samples, {phase_dfs[phase].shape[1]} columns")
    else:
        print(f"{phase}: Not available")

print(f"\nTotal phases loaded: {len(phase_dfs)}")

P-A: Not available
P-B: Not available
P-C: Not available

Total phases loaded: 0


### Data Quality Assessment

Identify and count rows with empty predictions or errors. These will be excluded from all metric calculations.

In [10]:
# Function to identify valid rows (exclude empty predictions and errors)
def is_valid_row(row, pred_col, err_col=None):
    """Check if a row has valid prediction (non-empty, no error)."""
    # Check if prediction is empty
    pred_value = row[pred_col]
    if pd.isna(pred_value) or str(pred_value).strip() == "":
        return False
    
    # Check if there's an error column and it has an error
    if err_col and err_col in row.index:
        if pd.notna(row[err_col]) and str(row[err_col]).strip() != "":
            return False
    
    return True

# Assess data quality for all phases
print("=" * 120)
print("DATA QUALITY ASSESSMENT")
print("=" * 120)

quality_stats = []

for phase, df in phase_dfs.items():
    print(f"\n📊 {phase} - Total rows: {len(df)}")
    
    pred_cols = [col for col in df.columns if col.startswith('predicted_boxes_')]
    
    for pred_col in pred_cols:
        model = pred_col.replace('predicted_boxes_', '')
        err_col = f'error_{model}'
        
        # Count valid and invalid rows
        empty_count = 0
        error_count = 0
        valid_count = 0
        
        for _, row in df.iterrows():
            if is_valid_row(row, pred_col, err_col):
                valid_count += 1
            else:
                # Check what type of invalid
                if pd.isna(row[pred_col]) or str(row[pred_col]).strip() == "":
                    empty_count += 1
                elif err_col in df.columns and pd.notna(row[err_col]):
                    error_count += 1
        
        invalid_count = empty_count + error_count
        valid_pct = (valid_count / len(df)) * 100
        
        print(f"  {model}:")
        print(f"    ✅ Valid rows: {valid_count}/{len(df)} ({valid_pct:.1f}%)")
        if invalid_count > 0:
            print(f"    ⚠️  Empty predictions: {empty_count}")
            if error_count > 0:
                print(f"    ❌ Errors logged: {error_count}")
        
        quality_stats.append({
            'Phase': phase,
            'Model': model,
            'Total Rows': len(df),
            'Valid Rows': valid_count,
            'Empty Predictions': empty_count,
            'Errors': error_count,
            'Valid %': valid_pct
        })

# Create summary DataFrame
quality_df = pd.DataFrame(quality_stats)

print("\n" + "=" * 120)
print("QUALITY SUMMARY - Models with Issues")
print("=" * 120)

# Show only models with issues
if 'Valid %' in quality_df.columns:
    issues_df = quality_df[quality_df['Valid %'] < 100].sort_values('Valid %')
else:
    issues_df = pd.DataFrame()

if len(issues_df) > 0:
    display(issues_df)
    print(f"\n⚠️  Found {len(issues_df)} model-phase combinations with data quality issues")
    print("These rows will be EXCLUDED from all metric calculations")
else:
    print("✅ All models have 100% valid data - no filtering needed!")

print("\n" + "=" * 120)

DATA QUALITY ASSESSMENT

QUALITY SUMMARY - Models with Issues
✅ All models have 100% valid data - no filtering needed!



## 3. Data Preview

Quick look at 10 random samples showing ground truth boxes.

In [11]:
# Preview one phase
if 'P-A' in phase_dfs:
    df_preview = phase_dfs['P-A']
    
    # Get 10 random samples
    random_samples = df_preview.sample(n=min(10, len(df_preview)), random_state=42)
    
    # Show basic info
    columns_to_show = ['sample_id', 'image_path', 'ground_truth_boxes', 'phase']
    display_df = random_samples[columns_to_show].copy()
    
    # Truncate long JSON strings for readability
    display_df['ground_truth_boxes'] = display_df['ground_truth_boxes'].apply(
        lambda x: str(x)[:100] + '...' if len(str(x)) > 100 else x
    )
    
    print(f"\n{'='*100}")
    print(f"DATA PREVIEW: publaynet_full - P-A")
    print(f"Showing 10 random samples with ground truth boxes")
    print(f"{'='*100}\n")
    
    display(display_df)
    
    print(f"\nTotal samples in P-A: {len(df_preview)}")
    
    # Show available model columns
    pred_cols = [col for col in df_preview.columns if col.startswith('predicted_boxes_')]
    print(f"Available models: {', '.join([col.replace('predicted_boxes_', '') for col in pred_cols])}")
else:
    print("P-A phase not found")

P-A phase not found


## 4. Define Bounding Box Evaluation Metrics

Implement custom functions for evaluating bounding boxes using IoU (Intersection over Union).

In [12]:
def calculate_iou(box1: Dict, box2: Dict) -> float:
    """
    Calculate Intersection over Union between two boxes.
    Box format: {x, y, width, height, category, confidence}
    """
    # Extract coordinates
    x1_min, y1_min = box1['x'], box1['y']
    x1_max, y1_max = x1_min + box1['width'], y1_min + box1['height']
    
    x2_min, y2_min = box2['x'], box2['y']
    x2_max, y2_max = x2_min + box2['width'], y2_min + box2['height']
    
    # Calculate intersection
    inter_x_min = max(x1_min, x2_min)
    inter_y_min = max(y1_min, y2_min)
    inter_x_max = min(x1_max, x2_max)
    inter_y_max = min(y1_max, y2_max)
    
    if inter_x_max <= inter_x_min or inter_y_max <= inter_y_min:
        return 0.0
    
    inter_area = (inter_x_max - inter_x_min) * (inter_y_max - inter_y_min)
    
    # Calculate union
    box1_area = box1['width'] * box1['height']
    box2_area = box2['width'] * box2['height']
    union_area = box1_area + box2_area - inter_area
    
    if union_area == 0:
        return 0.0
    
    return inter_area / union_area


def match_boxes(pred_boxes: List[Dict], gt_boxes: List[Dict], iou_threshold: float = 0.5) -> Tuple[List, List, List]:
    """
    Match predicted boxes to ground truth using IoU threshold.
    Returns: (matched_pairs, unmatched_gt, unmatched_pred)
    """
    matched_pairs = []  # (pred_idx, gt_idx, iou)
    matched_gt = set()
    matched_pred = set()
    
    # Calculate IoU matrix
    iou_matrix = np.zeros((len(pred_boxes), len(gt_boxes)))
    for i, pred_box in enumerate(pred_boxes):
        for j, gt_box in enumerate(gt_boxes):
            iou_matrix[i, j] = calculate_iou(pred_box, gt_box)
    
    # Greedy matching: highest IoU first
    while True:
        if iou_matrix.size == 0:
            break
        
        max_iou_idx = np.unravel_index(iou_matrix.argmax(), iou_matrix.shape)
        max_iou = iou_matrix[max_iou_idx]
        
        if max_iou < iou_threshold:
            break
        
        pred_idx, gt_idx = max_iou_idx
        matched_pairs.append((pred_idx, gt_idx, max_iou))
        matched_pred.add(pred_idx)
        matched_gt.add(gt_idx)
        
        # Remove matched row and column
        iou_matrix[pred_idx, :] = 0
        iou_matrix[:, gt_idx] = 0
    
    unmatched_gt = [i for i in range(len(gt_boxes)) if i not in matched_gt]
    unmatched_pred = [i for i in range(len(pred_boxes)) if i not in matched_pred]
    
    return matched_pairs, unmatched_gt, unmatched_pred


def calculate_box_metrics(pred_boxes: List[Dict], gt_boxes: List[Dict], iou_threshold: float = 0.5) -> Dict:
    """
    Calculate precision, recall, F1 for box detection.
    Returns: {precision, recall, f1, avg_iou, tp, fp, fn}
    """
    if len(gt_boxes) == 0:
        return {
            'precision': 0.0 if len(pred_boxes) > 0 else 1.0,
            'recall': 1.0,
            'f1': 0.0 if len(pred_boxes) > 0 else 1.0,
            'avg_iou': 0.0,
            'tp': 0,
            'fp': len(pred_boxes),
            'fn': 0
        }
    
    if len(pred_boxes) == 0:
        return {
            'precision': 0.0,
            'recall': 0.0,
            'f1': 0.0,
            'avg_iou': 0.0,
            'tp': 0,
            'fp': 0,
            'fn': len(gt_boxes)
        }
    
    matched_pairs, unmatched_gt, unmatched_pred = match_boxes(pred_boxes, gt_boxes, iou_threshold)
    
    tp = len(matched_pairs)
    fp = len(unmatched_pred)
    fn = len(unmatched_gt)
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    avg_iou = np.mean([iou for _, _, iou in matched_pairs]) if matched_pairs else 0.0
    
    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'avg_iou': avg_iou,
        'tp': tp,
        'fp': fp,
        'fn': fn
    }


def calculate_category_accuracy(pred_boxes: List[Dict], gt_boxes: List[Dict], 
                               matched_pairs: List[Tuple]) -> float:
    """
    Calculate category classification accuracy for matched boxes.
    """
    if not matched_pairs:
        return 0.0
    
    correct = 0
    for pred_idx, gt_idx, _ in matched_pairs:
        pred_cat = pred_boxes[pred_idx].get('category', None)
        gt_cat = gt_boxes[gt_idx].get('category', None)
        if pred_cat == gt_cat:
            correct += 1
    
    return correct / len(matched_pairs) if matched_pairs else 0.0


# Test the functions
test_box1 = {'x': 0, 'y': 0, 'width': 100, 'height': 100, 'category': 1}
test_box2 = {'x': 50, 'y': 50, 'width': 100, 'height': 100, 'category': 1}
test_iou = calculate_iou(test_box1, test_box2)
print(f"Test IoU (50% overlap): {test_iou:.4f}")
print("✓ Bounding box metrics defined successfully!")

Test IoU (50% overlap): 0.1429
✓ Bounding box metrics defined successfully!


## 5. Parse Bounding Box Data

Extract JSON bounding box data from each model column and normalize format.

In [13]:
def parse_boxes(boxes_str: str) -> List[Dict]:
    """
    Parse JSON string to list of box dictionaries.
    Handle empty predictions and malformed JSON.
    """
    if pd.isna(boxes_str) or boxes_str is None or boxes_str == '':
        return []
    
    try:
        boxes = json.loads(boxes_str)
        if not isinstance(boxes, list):
            return []
        return boxes
    except (json.JSONDecodeError, TypeError):
        return []


def get_model_names_from_df(df: pd.DataFrame) -> List[str]:
    """Extract model names from predicted_boxes_ column prefixes."""
    models = set()
    for col in df.columns:
        if col.startswith('predicted_boxes_'):
            model = col.replace('predicted_boxes_', '')
            models.add(model)
    return sorted(list(models))


# Get model names for each phase
phase_models = {}
for phase, df in phase_dfs.items():
    models = get_model_names_from_df(df)
    phase_models[phase] = models
    phase_label = {
        'P-A': 'OCR Baseline',
        'P-B': 'VLM Baseline',
        'P-C': 'VLM + Task-Aware'
    }.get(phase, phase)
    print(f"{phase} ({phase_label}): {models}")

In [14]:
# Count boxes per category in ground truth
CATEGORY_NAMES = {
    1: 'text',
    2: 'title',
    3: 'list',
    4: 'table',
    5: 'figure'
}

def analyze_box_distribution(df: pd.DataFrame, phase_name: str):
    """Analyze distribution of box categories in ground truth."""
    category_counts = {cat: 0 for cat in CATEGORY_NAMES.values()}
    total_boxes = 0
    
    for _, row in df.iterrows():
        gt_boxes = parse_boxes(row['ground_truth_boxes'])
        for box in gt_boxes:
            cat_id = box.get('category', 0)
            cat_name = CATEGORY_NAMES.get(cat_id, 'unknown')
            category_counts[cat_name] = category_counts.get(cat_name, 0) + 1
            total_boxes += 1
    
    print(f"\n{phase_name} - Ground Truth Box Distribution:")
    print(f"Total boxes: {total_boxes}")
    for cat, count in sorted(category_counts.items()):
        pct = count / total_boxes * 100 if total_boxes > 0 else 0
        print(f"  {cat}: {count} ({pct:.1f}%)")

# Analyze each phase
for phase, df in phase_dfs.items():
    analyze_box_distribution(df, phase)

## 6. Calculate Metrics for Each Phase

Compute box detection metrics (precision, recall, F1, IoU) for all models.

In [15]:
def calculate_metrics_for_phase(df: pd.DataFrame, models: List[str], phase_name: str) -> pd.DataFrame:
    """Calculate box detection metrics for all models in a phase."""
    results = []
    
    for model in models:
        pred_col = f"predicted_boxes_{model}"
        time_col = f"inference_time_ms_{model}"
        err_col = f"error_{model}"
        
        if pred_col not in df.columns:
            continue
        
        # Filter to only valid rows (non-empty predictions, no errors)
        valid_rows = []
        for _, row in df.iterrows():
            if is_valid_row(row, pred_col, err_col):
                valid_rows.append(row)
        
        print(f"  {model}: Using {len(valid_rows)}/{len(df)} valid rows (excluding {len(df)-len(valid_rows)} empty/error rows)")
        
        for row in valid_rows:
            gt_boxes = parse_boxes(row.get('ground_truth_boxes', ''))
            pred_boxes = parse_boxes(row.get(pred_col, ''))
            
            # Skip if no ground truth
            if len(gt_boxes) == 0:
                continue
            
            # Calculate box detection metrics
            box_metrics = calculate_box_metrics(pred_boxes, gt_boxes)
            
            # Calculate category accuracy
            matched_pairs, _, _ = match_boxes(pred_boxes, gt_boxes)
            cat_accuracy = calculate_category_accuracy(pred_boxes, gt_boxes, matched_pairs)
            
            # Get inference time
            inference_time = row.get(time_col, np.nan)
            
            results.append({
                'sample_id': row['sample_id'],
                'model': model,
                'phase': phase_name,
                'gt_box_count': len(gt_boxes),
                'pred_box_count': len(pred_boxes),
                'precision': box_metrics['precision'],
                'recall': box_metrics['recall'],
                'f1': box_metrics['f1'],
                'avg_iou': box_metrics['avg_iou'],
                'category_accuracy': cat_accuracy,
                'tp': box_metrics['tp'],
                'fp': box_metrics['fp'],
                'fn': box_metrics['fn'],
                'has_prediction': len(pred_boxes) > 0,
                'inference_time_ms': inference_time,
                'valid_samples': len(valid_rows),
                'total_samples': len(df)
            })
    
    return pd.DataFrame(results)


# Calculate metrics for each phase
all_metrics = []

phase_labels = {
    'P-A': 'Phase P-A (OCR)',
    'P-B': 'Phase P-B (VLM)',
    'P-C': 'Phase P-C (VLM+)'
}

for phase, df in phase_dfs.items():
    models = phase_models.get(phase, [])
    if models:
        print(f"Calculating metrics for {phase}...")
        phase_metrics = calculate_metrics_for_phase(df, models, phase_labels.get(phase, phase))
        print(f"  Computed {len(phase_metrics)} measurements")
        all_metrics.append(phase_metrics)

# Combine all metrics
if all_metrics:
    all_metrics_df = pd.concat(all_metrics, ignore_index=True)
    print(f"\nTotal measurements: {len(all_metrics_df)}")
else:
    all_metrics_df = pd.DataFrame()
    print("No metrics calculated")

No metrics calculated


## 7. Generate Summary Statistics

Aggregate metrics for each model and phase.

In [16]:
def summarize_metrics(metrics_df: pd.DataFrame) -> pd.DataFrame:
    """Generate summary statistics for each model and phase."""
    if len(metrics_df) == 0:
        return pd.DataFrame()
    
    agg_dict = {
        'precision': ['mean', 'median', 'std'],
        'recall': ['mean', 'median', 'std'],
        'f1': ['mean', 'median', 'std', 'min', 'max'],
        'avg_iou': ['mean', 'median', 'std'],
        'category_accuracy': ['mean', 'median'],
        'has_prediction': ['sum', 'count'],
        'inference_time_ms': ['mean', 'median', 'min', 'max'],
        'gt_box_count': 'mean',
        'pred_box_count': 'mean',
        'valid_samples': 'first',
        'total_samples': 'first'
    }
    
    summary = metrics_df.groupby(['model', 'phase']).agg(agg_dict).round(4)
    
    # Flatten column names
    summary.columns = ['_'.join(col).strip() for col in summary.columns.values]
    summary['prediction_rate'] = (summary['has_prediction_sum'] / summary['has_prediction_count'] * 100).round(1)
    
    return summary.reset_index()


# Generate summary
if len(all_metrics_df) > 0:
    metrics_summary = summarize_metrics(all_metrics_df)
    print("Model Performance Summary:")
    print("=" * 100)
    display(metrics_summary)
    
    # Show data quality summary
    print("\n📊 Data Quality Summary:")
    quality_summary = metrics_summary[['model', 'phase', 'valid_samples_first', 'total_samples_first']].copy()
    quality_summary.columns = ['Model', 'Phase', 'Valid Samples', 'Total Samples']
    quality_summary['Valid %'] = (quality_summary['Valid Samples'] / quality_summary['Total Samples'] * 100).round(1)
    display(quality_summary)
    
    # Best model per phase
    print("\nBest Model per Phase (by F1):")
    for phase in metrics_summary['phase'].unique():
        phase_data = metrics_summary[metrics_summary['phase'] == phase]
        best_model = phase_data.loc[phase_data['f1_mean'].idxmax()]
        print(f"  {phase}: {best_model['model']} (F1={best_model['f1_mean']:.4f}, Valid samples={int(best_model['valid_samples_first'])}/{int(best_model['total_samples_first'])})")

## 8. Model Performance Comparison

Compare models across phases to identify best performers.

In [17]:
if len(all_metrics_df) > 0 and len(metrics_summary) > 0:
    # Create clean comparison table
    comparison_cols = ['model', 'phase', 'precision_mean', 'recall_mean', 'f1_mean', 
                      'avg_iou_mean', 'category_accuracy_mean', 'prediction_rate']
    comparison_df = metrics_summary[comparison_cols].copy()
    comparison_df.columns = ['Model', 'Phase', 'Precision', 'Recall', 'F1', 
                            'Avg IoU', 'Cat. Acc.', 'Pred Rate %']
    
    # Sort by F1 score
    comparison_df = comparison_df.sort_values('F1', ascending=False)
    print("Model Performance Comparison (sorted by F1 score):")
    print("=" * 100)
    display(comparison_df)

In [18]:
# Compare VLM performance across phases (Does prompting help?)
if len(all_metrics_df) > 0:
    vlm_phases = ['Phase P-B (VLM)', 'Phase P-C (VLM+)']
    vlm_metrics = all_metrics_df[all_metrics_df['phase'].isin(vlm_phases)]
    
    # Models in multiple phases
    model_phase_counts = vlm_metrics.groupby('model')['phase'].nunique()
    multi_phase_models = model_phase_counts[model_phase_counts > 1].index.tolist()
    
    print("Impact of Task-Aware Prompting on VLM Performance:")
    print("=" * 80)
    
    for model in sorted(multi_phase_models):
        print(f"\n{model}:")
        model_data = metrics_summary[metrics_summary['model'] == model]
        
        phases_present = model_data['phase'].tolist()
        for phase in phases_present:
            row = model_data[model_data['phase'] == phase].iloc[0]
            print(f"  {phase}: F1={row['f1_mean']:.4f}, Precision={row['precision_mean']:.4f}, Recall={row['recall_mean']:.4f}")
        
        # Calculate improvement from P-B to P-C if available
        if 'Phase P-B (VLM)' in phases_present and 'Phase P-C (VLM+)' in phases_present:
            pb_f1 = model_data[model_data['phase'] == 'Phase P-B (VLM)']['f1_mean'].values[0]
            pc_f1 = model_data[model_data['phase'] == 'Phase P-C (VLM+)']['f1_mean'].values[0]
            improvement = ((pc_f1 - pb_f1) / pb_f1 * 100) if pb_f1 > 0 else 0
            if improvement > 0:
                print(f"  ✓ Task-aware prompting IMPROVED F1 by {improvement:.1f}% (P-B → P-C)")
            elif improvement < 0:
                print(f"  ✗ Task-aware prompting WORSENED F1 by {abs(improvement):.1f}% (P-B → P-C)")
            else:
                print(f"  = Prompting had NO EFFECT on F1")

## 9. Visualizations

Generate charts to compare model performance.

In [19]:
if len(all_metrics_df) > 0 and len(metrics_summary) > 0:
    # Bar charts: Precision, Recall, F1 by Model
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Color scheme for phases
    phase_colors = {
        'Phase P-A (OCR)': '#2ecc71',
        'Phase P-B (VLM)': '#3498db',
        'Phase P-C (VLM+)': '#9b59b6'
    }
    
    x_positions = range(len(metrics_summary))
    
    # Plot 1: Precision
    ax1 = axes[0]
    bars = ax1.bar(x_positions, metrics_summary['precision_mean'], 
                   color=[phase_colors.get(p, '#95a5a6') for p in metrics_summary['phase']])
    ax1.set_xticks(x_positions)
    ax1.set_xticklabels([f"{row['model']}\n({row['phase'].split()[1]})" 
                          for _, row in metrics_summary.iterrows()], rotation=45, ha='right')
    ax1.set_ylabel('Precision')
    ax1.set_title('Precision by Model (Higher is Better)')
    ax1.set_ylim(0, 1.0)
    
    # Plot 2: Recall
    ax2 = axes[1]
    bars = ax2.bar(x_positions, metrics_summary['recall_mean'],
                   color=[phase_colors.get(p, '#95a5a6') for p in metrics_summary['phase']])
    ax2.set_xticks(x_positions)
    ax2.set_xticklabels([f"{row['model']}\n({row['phase'].split()[1]})" 
                          for _, row in metrics_summary.iterrows()], rotation=45, ha='right')
    ax2.set_ylabel('Recall')
    ax2.set_title('Recall by Model (Higher is Better)')
    ax2.set_ylim(0, 1.0)
    
    # Plot 3: F1
    ax3 = axes[2]
    bars = ax3.bar(x_positions, metrics_summary['f1_mean'],
                   color=[phase_colors.get(p, '#95a5a6') for p in metrics_summary['phase']])
    ax3.set_xticks(x_positions)
    ax3.set_xticklabels([f"{row['model']}\n({row['phase'].split()[1]})" 
                          for _, row in metrics_summary.iterrows()], rotation=45, ha='right')
    ax3.set_ylabel('F1 Score')
    ax3.set_title('F1 Score by Model (Higher is Better)')
    ax3.set_ylim(0, 1.0)
    
    plt.tight_layout()
    plt.savefig('../../../4_postprocessing/publaynet_precision_recall_f1.png', dpi=150, bbox_inches='tight')
    plt.show()

In [20]:
if len(all_metrics_df) > 0:
    # Box plots: F1 and IoU distribution by model
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Get models ordered by mean F1
    models_ordered = metrics_summary.sort_values('f1_mean', ascending=False)['model'].unique().tolist()
    
    # Box plot for F1
    ax1 = axes[0]
    box_data_f1 = [all_metrics_df[all_metrics_df['model'] == m]['f1'].values for m in models_ordered]
    bp1 = ax1.boxplot(box_data_f1, labels=models_ordered, patch_artist=True)
    
    # Color boxes by phase
    for i, model in enumerate(models_ordered):
        phase = metrics_summary[metrics_summary['model'] == model]['phase'].values[0]
        bp1['boxes'][i].set_facecolor(phase_colors.get(phase, '#95a5a6'))
    
    ax1.set_ylabel('F1 Score')
    ax1.set_title('F1 Score Distribution by Model')
    ax1.tick_params(axis='x', rotation=45)
    ax1.set_ylim(0, 1.0)
    
    # Box plot for IoU
    ax2 = axes[1]
    box_data_iou = [all_metrics_df[all_metrics_df['model'] == m]['avg_iou'].values for m in models_ordered]
    bp2 = ax2.boxplot(box_data_iou, labels=models_ordered, patch_artist=True)
    
    for i, model in enumerate(models_ordered):
        phase = metrics_summary[metrics_summary['model'] == model]['phase'].values[0]
        bp2['boxes'][i].set_facecolor(phase_colors.get(phase, '#95a5a6'))
    
    ax2.set_ylabel('Average IoU')
    ax2.set_title('IoU Distribution by Model')
    ax2.tick_params(axis='x', rotation=45)
    ax2.set_ylim(0, 1.0)
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=c, label=p) for p, c in phase_colors.items() 
                       if p in metrics_summary['phase'].values]
    fig.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(0.99, 0.99))
    
    plt.tight_layout()
    plt.savefig('../../../4_postprocessing/publaynet_f1_iou_boxplot.png', dpi=150, bbox_inches='tight')
    plt.show()

## 10. Category-Level Analysis

Analyze performance by document element type (text, title, list, table, figure).

In [21]:
def analyze_category_performance(df: pd.DataFrame, models: List[str], phase_name: str) -> pd.DataFrame:
    """Analyze detection performance by category."""
    results = []
    
    for model in models:
        pred_col = f"predicted_boxes_{model}"
        
        if pred_col not in df.columns:
            continue
        
        # Track metrics per category
        category_metrics = {cat: {'tp': 0, 'fp': 0, 'fn': 0} for cat in CATEGORY_NAMES.values()}
        
        for _, row in df.iterrows():
            gt_boxes = parse_boxes(row.get('ground_truth_boxes', ''))
            pred_boxes = parse_boxes(row.get(pred_col, ''))
            
            if len(gt_boxes) == 0:
                continue
            
            matched_pairs, unmatched_gt, unmatched_pred = match_boxes(pred_boxes, gt_boxes)
            
            # True positives (matched with correct category)
            for pred_idx, gt_idx, _ in matched_pairs:
                gt_cat = CATEGORY_NAMES.get(gt_boxes[gt_idx].get('category', 0), 'unknown')
                pred_cat = CATEGORY_NAMES.get(pred_boxes[pred_idx].get('category', 0), 'unknown')
                
                if gt_cat == pred_cat:
                    category_metrics[gt_cat]['tp'] += 1
                else:
                    category_metrics[gt_cat]['fn'] += 1
                    category_metrics[pred_cat]['fp'] += 1
            
            # False negatives (unmatched ground truth)
            for gt_idx in unmatched_gt:
                gt_cat = CATEGORY_NAMES.get(gt_boxes[gt_idx].get('category', 0), 'unknown')
                category_metrics[gt_cat]['fn'] += 1
            
            # False positives (unmatched predictions)
            for pred_idx in unmatched_pred:
                pred_cat = CATEGORY_NAMES.get(pred_boxes[pred_idx].get('category', 0), 'unknown')
                category_metrics[pred_cat]['fp'] += 1
        
        # Calculate metrics for each category
        for category, counts in category_metrics.items():
            tp, fp, fn = counts['tp'], counts['fp'], counts['fn']
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
            
            results.append({
                'model': model,
                'phase': phase_name,
                'category': category,
                'precision': precision,
                'recall': recall,
                'f1': f1,
                'tp': tp,
                'fp': fp,
                'fn': fn
            })
    
    return pd.DataFrame(results)


# Analyze categories for each phase
all_category_metrics = []

for phase, df in phase_dfs.items():
    models = phase_models.get(phase, [])
    if models:
        print(f"Analyzing categories for {phase}...")
        cat_metrics = analyze_category_performance(df, models, phase_labels.get(phase, phase))
        all_category_metrics.append(cat_metrics)

if all_category_metrics:
    category_df = pd.concat(all_category_metrics, ignore_index=True)
    print(f"\nCategory-level metrics calculated for {len(category_df)} entries")
    
    # Display summary
    print("\nCategory Performance Summary (F1 scores):")
    print("=" * 80)
    pivot = category_df.pivot_table(index='category', columns=['model', 'phase'], values='f1')
    display(pivot.round(4))

In [22]:
# Visualize category performance
if len(all_category_metrics) > 0:
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # Group by category and calculate mean F1
    cat_summary = category_df.groupby(['category', 'model'])['f1'].mean().reset_index()
    
    # Create grouped bar chart
    categories = sorted(cat_summary['category'].unique())
    models = sorted(cat_summary['model'].unique())
    
    x = np.arange(len(categories))
    width = 0.8 / len(models)
    
    for i, model in enumerate(models):
        model_data = cat_summary[cat_summary['model'] == model]
        f1_values = [model_data[model_data['category'] == cat]['f1'].values[0] 
                     if cat in model_data['category'].values else 0 
                     for cat in categories]
        ax.bar(x + i * width, f1_values, width, label=model, alpha=0.8)
    
    ax.set_xlabel('Document Element Category')
    ax.set_ylabel('F1 Score')
    ax.set_title('Detection Performance by Category')
    ax.set_xticks(x + width * (len(models) - 1) / 2)
    ax.set_xticklabels(categories)
    ax.legend(loc='best')
    ax.set_ylim(0, 1.0)
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../../../4_postprocessing/publaynet_category_performance.png', dpi=150, bbox_inches='tight')
    plt.show()

## 11. Sample-Level Analysis

Identify easiest and hardest samples for models.

In [23]:
if len(all_metrics_df) > 0:
    # Pivot to compare same samples across models
    sample_pivot = all_metrics_df.pivot_table(
        index='sample_id', 
        columns='model', 
        values='f1',
        aggfunc='first'
    ).reset_index()
    
    # Calculate mean F1 across all models for each sample
    model_cols = [c for c in sample_pivot.columns if c != 'sample_id']
    sample_pivot['mean_f1'] = sample_pivot[model_cols].mean(axis=1)
    sample_pivot['std_f1'] = sample_pivot[model_cols].std(axis=1)
    
    # Best samples (highest mean F1)
    print("Top 10 EASIEST Samples (highest mean F1 across all models):")
    print("=" * 80)
    print(sample_pivot.nlargest(10, 'mean_f1')[['sample_id', 'mean_f1', 'std_f1']].to_string(index=False))
    
    print("\n")
    print("Top 10 HARDEST Samples (lowest mean F1 across all models):")
    print("=" * 80)
    print(sample_pivot.nsmallest(10, 'mean_f1')[['sample_id', 'mean_f1', 'std_f1']].to_string(index=False))

In [24]:
# Analyze over-detection and under-detection patterns
if len(all_metrics_df) > 0:
    print("\nBox Count Analysis (Ground Truth vs Predictions):")
    print("=" * 80)
    
    for model in sorted(all_metrics_df['model'].unique()):
        model_data = all_metrics_df[all_metrics_df['model'] == model]
        
        avg_gt = model_data['gt_box_count'].mean()
        avg_pred = model_data['pred_box_count'].mean()
        over_detect_pct = (avg_pred / avg_gt - 1) * 100 if avg_gt > 0 else 0
        
        print(f"\n{model}:")
        print(f"  Avg GT boxes: {avg_gt:.1f}")
        print(f"  Avg predicted boxes: {avg_pred:.1f}")
        
        if over_detect_pct > 5:
            print(f"  ⚠️ Over-detection: {over_detect_pct:.1f}% more boxes than ground truth")
        elif over_detect_pct < -5:
            print(f"  ⚠️ Under-detection: {abs(over_detect_pct):.1f}% fewer boxes than ground truth")
        else:
            print(f"  ✓ Balanced detection: {over_detect_pct:+.1f}% vs ground truth")

## 12. Inference Time Analysis

Compare inference times and analyze speed vs accuracy tradeoffs.

In [25]:
if len(all_metrics_df) > 0 and 'inference_time_ms' in all_metrics_df.columns:
    # Filter out missing times
    time_data = all_metrics_df.dropna(subset=['inference_time_ms'])
    
    if len(time_data) > 0:
        # Summary by model
        time_summary = time_data.groupby(['model', 'phase']).agg({
            'inference_time_ms': ['mean', 'median', 'min', 'max'],
            'f1': 'mean'
        }).round(2)
        
        time_summary.columns = ['_'.join(col) for col in time_summary.columns]
        time_summary = time_summary.reset_index()
        
        print("Inference Time Summary:")
        print("=" * 80)
        display(time_summary)
        
        # Visualize
        fig, ax = plt.subplots(figsize=(12, 6))
        
        x = range(len(time_summary))
        bars = ax.bar(x, time_summary['inference_time_ms_mean'])
        
        ax.set_xticks(x)
        ax.set_xticklabels([f"{row['model']}\n({row['phase'].split()[1]})" 
                           for _, row in time_summary.iterrows()], rotation=45, ha='right')
        ax.set_ylabel('Average Inference Time (ms)')
        ax.set_title('Inference Time by Model and Phase')
        
        # Add value labels
        for bar, val in zip(bars, time_summary['inference_time_ms_mean']):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10, 
                    f'{val:.0f}', ha='center', va='bottom', fontsize=9)
        
        plt.tight_layout()
        plt.savefig('../../../4_postprocessing/publaynet_inference_time.png', dpi=150, bbox_inches='tight')
        plt.show()
else:
    print("No inference time data available")

No inference time data available


## 13. LLM Query Section

This section is a placeholder for analyzing the notebook outputs using an LLM.

### Key Questions to Investigate:

1. **Performance Comparison:**
   - Which model performs best overall for document layout detection?
   - How do OCR models (P-A) compare to VLMs (P-B, P-C)?
   - What is the impact of task-aware prompting (P-C vs P-B)?

2. **Metric Insights:**
   - Is there a strong correlation between Precision, Recall, and F1?
   - Do models with high F1 also have high IoU scores?
   - Are there models that excel at one metric but not others?

3. **Category-Specific Performance:**
   - Which document elements (text, title, table, figure, list) are easiest/hardest to detect?
   - Do different models excel at different categories?
   - Does task-aware prompting help with specific categories more than others?

4. **Detection Patterns:**
   - Do models tend to over-detect or under-detect bounding boxes?
   - What are the most common failure modes?
   - Are there systematic errors in category classification?

5. **Speed vs Accuracy Trade-offs:**
   - Which model offers the best balance between inference time and detection quality?
   - Is the speed difference justified by accuracy improvements?

6. **Recommendations:**
   - Which model should be used for production document layout detection?
   - Are specialized OCR models still necessary, or can VLMs replace them?
   - What improvements could be made to prompting strategies?

### Important Note:
**No cosine similarity analysis** - This is a bounding box detection task, not a text extraction task. Semantic similarity metrics are not applicable to spatial layout detection.

In [26]:
# Speed vs Quality scatterplot
if len(all_metrics_df) > 0 and 'inference_time_ms' in all_metrics_df.columns:
    time_data = all_metrics_df.dropna(subset=['inference_time_ms'])
    
    if len(time_data) > 0:
        # Calculate average F1 and inference time per model
        model_summary = time_data.groupby('model').agg({
            'f1': 'mean',
            'inference_time_ms': 'mean'
        }).reset_index()
        
        fig, ax = plt.subplots(figsize=(10, 6))
        
        ax.scatter(model_summary['inference_time_ms'], model_summary['f1'], 
                   s=200, alpha=0.6, c=range(len(model_summary)), cmap='viridis')
        
        # Add model labels
        for _, row in model_summary.iterrows():
            ax.annotate(row['model'], 
                       (row['inference_time_ms'], row['f1']),
                       xytext=(5, 5), textcoords='offset points', fontsize=9)
        
        ax.set_xlabel('Average Inference Time (ms)')
        ax.set_ylabel('Average F1 Score')
        ax.set_title('Speed vs Quality Tradeoff (Lower time, Higher F1 is better)')
        ax.grid(alpha=0.3)
        
        plt.tight_layout()
        plt.savefig('../../../4_postprocessing/publaynet_speed_quality_tradeoff.png', dpi=150, bbox_inches='tight')
        plt.show()

## 14. Export Results

Save analysis outputs for further use.

In [27]:
# Create output directory if needed
output_dir = Path('../../../4_postprocessing/publaynet_full')
output_dir.mkdir(parents=True, exist_ok=True)

if len(all_metrics_df) > 0:
    # Save detailed metrics
    all_metrics_df.to_csv(output_dir / 'detailed_metrics.csv', index=False)
    print(f"Saved detailed metrics to: {output_dir / 'detailed_metrics.csv'}")
    
    # Save model comparison
    if 'comparison_df' in dir() and len(comparison_df) > 0:
        comparison_df.to_csv(output_dir / 'model_comparison.csv', index=False)
        print(f"Saved model comparison to: {output_dir / 'model_comparison.csv'}")
    
    # Save category breakdown
    if 'category_df' in dir() and len(category_df) > 0:
        category_df.to_csv(output_dir / 'category_breakdown.csv', index=False)
        print(f"Saved category breakdown to: {output_dir / 'category_breakdown.csv'}")
    
    # Save sample-level analysis
    if 'sample_pivot' in dir():
        sample_pivot.to_csv(output_dir / 'sample_level_f1.csv', index=False)
        print(f"Saved sample-level F1 to: {output_dir / 'sample_level_f1.csv'}")
    
    print("\nAll exports complete!")
    print(f"Generated plots saved to: ../../../4_postprocessing/")
else:
    print("No data to export.")

No data to export.


## 15. Summary & Conclusions

### Key Findings:

1. **OCR vs VLM Performance**: 
   - Compare the F1, Precision, and Recall metrics between Phase P-A (OCR) and Phase P-B/P-C (VLM) models for document layout detection.
   - OCR models are specialized for layout detection, but VLMs offer more flexibility.

2. **Impact of Task-Aware Prompting**: 
   - Compare Phase P-B (generic prompts) vs Phase P-C (task-specific prompts) to determine if detailed instructions improve VLM performance.
   - Analyze whether prompting helps with specific categories (e.g., tables vs text).

3. **Best Performing Models**: 
   - Identify which models achieve the highest F1 scores for bounding box detection.
   - Consider speed vs accuracy tradeoffs.

4. **Category-Specific Performance**: 
   - Determine which document elements (text, title, table, figure, list) are easiest/hardest to detect.
   - Some categories may benefit more from VLM reasoning (e.g., distinguishing title from text).

5. **Detection Patterns**:
   - Analyze whether models tend to over-detect or under-detect boxes.
   - Identify common failure modes (missing small elements, merging adjacent boxes, etc.).

### Special Considerations for Document Layout Detection:
- IoU threshold of 0.5 is standard for object detection tasks
- Category accuracy is crucial - a detected box with wrong category is not useful
- Layout understanding requires spatial reasoning, not just text extraction
- Tables and figures are typically harder to detect than text blocks
- **No cosine similarity metrics** - This is a bounding box detection task, not a text extraction task

### Next Steps:
- Test additional VLM models to expand comparison
- Experiment with different IoU thresholds
- Analyze failure cases in detail (visualize predictions)
- Compare with specialized layout detection models (LayoutLM, DETR-based)
- Evaluate on more diverse document types (papers, forms, magazines)